# 08 - Predicción manual Top25 con scaler

Este notebook permite ingresar manualmente los valores de las **25 características** utilizadas por el modelo, aplicar el mismo escalado del proyecto y obtener la predicción.

Es especialmente útil para asignar manualmente `google_index` cuando la consulta automática a Google responde con HTTP 429.

**Convención esperada del modelo:**

- `prediction = 1` → phishing
- `prediction = 0` → no phishing
- `google_index = 1` → el dominio aparece indexado
- `google_index = 0` → el dominio no aparece indexado

> No uses `-1` en `google_index` salvo que el modelo haya sido reentrenado con ese valor.


In [1]:
import os
import sys
import importlib
from pathlib import Path

import pandas as pd
import numpy as np
import sklearn

sys.dont_write_bytecode = True
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"

# Busca automáticamente la carpeta principal del proyecto.
BASE_DIR = Path.cwd()

if not (BASE_DIR / "feature_extractor_top25.py").exists():
    BASE_DIR = Path(r"C:\phishing-url-classifier\phishing-url-ai")

if not (BASE_DIR / "feature_extractor_top25.py").exists():
    raise FileNotFoundError(
        "No se encontró feature_extractor_top25.py. "
        "Abre este notebook desde la carpeta phishing-url-ai "
        "o modifica manualmente BASE_DIR."
    )

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

print("BASE_DIR:", BASE_DIR)
print("scikit-learn:", sklearn.__version__)

BASE_DIR: c:\phishing-url-classifier\phishing-url-ai
scikit-learn: 1.7.2


In [2]:
import feature_extractor_top25 as extractor
extractor = importlib.reload(extractor)

load_feature_names = extractor.load_feature_names
load_continuous_features = extractor.load_continuous_features
scale_top25_features = extractor.scale_top25_features
load_model = extractor.load_model

MODEL_FEATURES = list(load_feature_names())
CONTINUOUS_FEATURES = list(load_continuous_features())
DISCRETE_FEATURES = [
    feature for feature in MODEL_FEATURES
    if feature not in CONTINUOUS_FEATURES
]

model = load_model()

print("Extractor:", extractor.__file__)
print("Cantidad total de variables:", len(MODEL_FEATURES))
print("Variables continuas:", len(CONTINUOUS_FEATURES))
print("Variables discretas:", len(DISCRETE_FEATURES))
print("Clases del modelo:", list(model.classes_))

if hasattr(model, "feature_names_in_"):
    assert list(model.feature_names_in_) == MODEL_FEATURES, (
        "El orden del modelo no coincide con MODEL_FEATURES."
    )

display(pd.DataFrame({
    "posición": range(1, len(MODEL_FEATURES) + 1),
    "feature": MODEL_FEATURES,
    "tipo": [
        "continua" if feature in CONTINUOUS_FEATURES else "discreta"
        for feature in MODEL_FEATURES
    ]
}))

Extractor: c:\phishing-url-classifier\phishing-url-ai\feature_extractor_top25.py
Cantidad total de variables: 25
Variables continuas: 20
Variables discretas: 5
Clases del modelo: [0, 1]


,posición,feature,tipo
0,1,google_index,discreta
1,2,domain_age,continua
2,3,nb_hyperlinks,continua
3,4,nb_www,discreta
4,5,phish_hints,continua
5,6,safe_anchor,continua
6,7,longest_word_path,continua
7,8,ratio_digits_url,continua
8,9,ratio_intHyperlinks,continua
9,10,longest_words_raw,continua


## 1. Ingresa manualmente los valores

Modifica solamente los números del diccionario. Los nombres y el orden se validan automáticamente.

Puedes comenzar con todos los valores en cero o copiar valores obtenidos previamente por el extractor y cambiar únicamente aquellos que necesites, como `google_index`.


In [3]:
# Identificador opcional para reconocer la prueba.
URL_REFERENCIA = "https://ejemplo.com"

# Modifica estos valores manualmente.
VALORES_MANUALES = {
    "domain_age": 10525,
    "nb_hyperlinks": 11,
    "phish_hints": 0,
    "safe_anchor": 0,
    "longest_word_path": 0,
    "ratio_digits_url": 0,
    "ratio_intHyperlinks": 0.909091,
    "longest_words_raw": 6,
    "length_words_raw": 12,
    "nb_dots": 2,
    "domain_registration_length": 796,
    "ratio_extRedirection": 0,
    "nb_hyphens": 0,
    "char_repeat": 5,
    "length_hostname": 14,
    "nb_slash": 2,
    "ratio_extHyperlinks": 0.090909,
    "links_in_tags": 3,
    "avg_word_path": 0,
    "ratio_extMedia": 0,

    # Variables discretas
    "google_index": 1,
    "nb_www": 1,
    "domain_in_title": 1,
    "domain_with_copyright": 1,
    "ip": 0,
}

# Ejemplo: marcar manualmente que el dominio sí está indexado.
# VALORES_MANUALES["google_index"] = 1


## 2. Validar, ordenar y visualizar los valores

Esta celda comprueba que:

- no falten características;
- no existan características desconocidas;
- todos los valores sean numéricos;
- el orden sea exactamente el mismo utilizado durante el entrenamiento.


In [4]:
missing = [feature for feature in MODEL_FEATURES if feature not in VALORES_MANUALES]
extra = [feature for feature in VALORES_MANUALES if feature not in MODEL_FEATURES]

if missing:
    raise ValueError(f"Faltan características: {missing}")

if extra:
    raise ValueError(f"Hay características desconocidas: {extra}")

manual_features = pd.DataFrame(
    [[VALORES_MANUALES[feature] for feature in MODEL_FEATURES]],
    columns=MODEL_FEATURES,
    dtype=float,
)

if manual_features.isna().any().any():
    raise ValueError("Existen valores vacíos o no numéricos.")

assert list(manual_features.columns) == MODEL_FEATURES
assert manual_features.shape == (1, len(MODEL_FEATURES))

print("Valores manuales validados correctamente.")
display(manual_features.T.rename(columns={0: "valor_manual"}))

Valores manuales validados correctamente.


,valor_manual
google_index,1.000000
domain_age,10525.000000
nb_hyperlinks,11.000000
nb_www,1.000000
phish_hints,0.000000
safe_anchor,0.000000
longest_word_path,0.000000
ratio_digits_url,0.000000
ratio_intHyperlinks,0.909091
longest_words_raw,6.000000


## 3. Aplicar el scaler

Solo se escalan las variables continuas definidas en el proyecto. Las variables discretas se conservan con sus valores originales.


In [5]:
scaled_features = scale_top25_features(manual_features.copy())

assert list(scaled_features.columns) == MODEL_FEATURES
assert scaled_features.shape == manual_features.shape
assert scaled_features.isna().sum().sum() == 0

comparison = pd.DataFrame({
    "valor_manual": manual_features.iloc[0],
    "valor_escalado": scaled_features.iloc[0],
    "tipo": [
        "continua" if feature in CONTINUOUS_FEATURES else "discreta"
        for feature in MODEL_FEATURES
    ],
})

display(comparison)

,valor_manual,valor_escalado,tipo
google_index,1.000000,1.000000,discreta
domain_age,10525.000000,2.079508,continua
nb_hyperlinks,11.000000,-0.456925,continua
nb_www,1.000000,1.000000,discreta
phish_hints,0.000000,-0.388992,continua
safe_anchor,0.000000,-0.948603,continua
longest_word_path,0.000000,-0.457664,continua
ratio_digits_url,0.000000,-0.594606,continua
ratio_intHyperlinks,0.909091,0.814434,continua
longest_words_raw,6.000000,-0.425404,continua


## 4. Realizar la predicción manual

In [6]:
prediction = int(model.predict(scaled_features)[0])
label = "phishing" if prediction == 1 else "no-phishing"

probability_phishing = None
probability_no_phishing = None

if hasattr(model, "predict_proba"):
    classes = list(model.classes_)
    probabilities = model.predict_proba(scaled_features)[0]

    probability_by_class = {
        int(class_value): float(probability)
        for class_value, probability in zip(classes, probabilities)
    }

    probability_no_phishing = probability_by_class.get(0)
    probability_phishing = probability_by_class.get(1)

resultado = pd.DataFrame([{
    "url_referencia": URL_REFERENCIA,
    "prediction": prediction,
    "clasificación": label,
    "probabilidad_no_phishing": probability_no_phishing,
    "probabilidad_phishing": probability_phishing,
}])

display(resultado)

print(f"Resultado: {label}")
if probability_phishing is not None:
    print(f"Probabilidad de phishing: {probability_phishing:.2%}")
if probability_no_phishing is not None:
    print(f"Probabilidad de no phishing: {probability_no_phishing:.2%}")

,url_referencia,prediction,clasificación,probabilidad_no_phishing,probabilidad_phishing
0,https://ejemplo.com,0,no-phishing,0.796667,0.203333


Resultado: no-phishing
Probabilidad de phishing: 20.33%
Probabilidad de no phishing: 79.67%


## 5. Opción práctica: extraer automáticamente y sobrescribir valores

Esta sección es opcional. Permite extraer las características de una URL y después reemplazar manualmente una o varias variables antes de predecir.

Por ejemplo, puedes dejar que el extractor calcule todo y cambiar únicamente:

```python
SOBRESCRIBIR = {"google_index": 1}
```


In [7]:
# Ejecuta esta celda solo cuando quieras partir desde una extracción automática.
URL_AUTOMATICA = "https://www.google.com"
USE_NETWORK_FEATURES = True

raw_automatic = extractor.extract_top25_features(
    URL_AUTOMATICA,
    use_network_features=USE_NETWORK_FEATURES,
)

# Aquí puedes reemplazar una o varias variables manualmente.
SOBRESCRIBIR = {
    "google_index": 1,
}

for feature, value in SOBRESCRIBIR.items():
    if feature not in MODEL_FEATURES:
        raise KeyError(f"La variable '{feature}' no pertenece al modelo Top25.")
    raw_automatic.loc[0, feature] = float(value)

raw_automatic = raw_automatic[MODEL_FEATURES]
scaled_automatic = scale_top25_features(raw_automatic.copy())

prediction_automatic = int(model.predict(scaled_automatic)[0])
label_automatic = "phishing" if prediction_automatic == 1 else "no-phishing"

probability_phishing_automatic = None
if hasattr(model, "predict_proba"):
    classes = list(model.classes_)
    probabilities = model.predict_proba(scaled_automatic)[0]
    if 1 in classes:
        probability_phishing_automatic = float(probabilities[classes.index(1)])

print("URL:", URL_AUTOMATICA)
print("Valores sobrescritos:", SOBRESCRIBIR)
print("Predicción:", prediction_automatic)
print("Clasificación:", label_automatic)

if probability_phishing_automatic is not None:
    print(f"Probabilidad de phishing: {probability_phishing_automatic:.2%}")

display(raw_automatic.T.rename(columns={0: "valor_final_sin_escalar"}))

URL: https://www.google.com
Valores sobrescritos: {'google_index': 1}
Predicción: 0
Clasificación: no-phishing
Probabilidad de phishing: 20.33%


,valor_final_sin_escalar
google_index,1.000000
domain_age,10526.000000
nb_hyperlinks,11.000000
nb_www,1.000000
phish_hints,0.000000
safe_anchor,0.000000
longest_word_path,0.000000
ratio_digits_url,0.000000
ratio_intHyperlinks,0.909091
longest_words_raw,6.000000
